In [3]:
import time
import math
import requests
from bs4 import BeautifulSoup
import pandas as pd
from collections import namedtuple
import re
#from sentence_transformers import SentenceTransformer, util
import time
from tqdm import tqdm
from collections import namedtuple
import random
import numpy as np


import json
import requests
from bs4 import BeautifulSoup

In [1]:
import json
import re
import requests
from bs4 import BeautifulSoup

def get_lat_lon(url, headers):
    r = requests.get(url, headers=headers, timeout=20)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")
    s = soup.find("script", id="__PRELOADED_STATE__")
    if not s:
        return None, None

    raw = s.get_text(strip=False)

    # recortamos al JSON grande
    m = re.search(r"\{.*\}", raw, flags=re.DOTALL)
    if not m:
        return None, None

    data = json.loads(m.group(0))

    # 🔎 búsqueda DIRIGIDA de map_info → location
    def find_map_info(obj):
        if isinstance(obj, dict):
            if "map_info" in obj:
                loc = obj["map_info"].get("location", {})
                lat = loc.get("latitude")
                lon = loc.get("longitude")
                if lat is not None and lon is not None:
                    return lat, lon
            for v in obj.values():
                res = find_map_info(v)
                if res != (None, None):
                    return res

        elif isinstance(obj, list):
            for it in obj:
                res = find_map_info(it)
                if res != (None, None):
                    return res

        return None, None

    return find_map_info(data)


In [4]:
def limpia_precio(txt):
    if not txt: 
        return None
    # quita símbolos comunes de MercadoLibre (puntos de miles, comas, $)
    t = txt.replace("\xa0", " ").replace(".", "").replace(",", "").replace("$", "").strip()
    # a veces viene junto con centavos en otro span; nos quedamos con entero
    try:
        return int("".join(ch for ch in t if ch.isdigit()))
    except ValueError:
        return None
import re

def extrae_atributos(li):
    """
    Regresa dict con recamaras, banos, sup_m2, sup_construida_m2 (si aparece), etc.
    Toma los <li> dentro de ul.poly-attributes_list
    """
    attrs = {
        "recamaras": None,
        "banos": None,
        "superficie_m2": None,
        "superficie_construida_m2": None
    }

    items = li.select("ul.poly-attributes_list li.poly-attributes_list__item")
    textos = [it.get_text(" ", strip=True).lower() for it in items]

    for t in textos:
        # recámaras / habitaciones / dormitorios
        m = re.search(r"(\d+)\s*(recámaras|recamaras|habitaciones|dormitorios)", t)
        if m:
            attrs["recamaras"] = int(m.group(1))
            continue

        # baños
        m = re.search(r"(\d+)\s*(baños|banos)", t)
        if m:
            attrs["banos"] = int(m.group(1))
            continue

        # superficie construida
        m = re.search(r"(\d+(?:\.\d+)?)\s*m²\s*(construidos|construidas|construido|construida)", t)
        if m:
            attrs["superficie_construida_m2"] = float(m.group(1))
            continue

        # superficie (genérica) en m²
        m = re.search(r"(\d+(?:\.\d+)?)\s*m²", t)
        if m and attrs["superficie_m2"] is None:
            attrs["superficie_m2"] = float(m.group(1))
            continue

    return attrs

def parse_item(li):
    # Títulos
    nombre = None
    brand_span = li.select_one(".poly-component__brand")
    if brand_span:
        nombre = brand_span.get_text(strip=True)
        
    descripcion = None
    t = li.find("h2")
    if t:
        descripcion = t.get_text(strip=True)
    else:
        t = li.find("h3")
        if t: descripcion = t.get_text(strip=True)

    # Link
    a = li.find("a", href=True)
    link = a["href"] if a else None

    # Imagen
    img = li.find("img")
    imagen = img.get("data-src") or img.get("src") if img else None

    # Precios (ML usa varias clases; probamos alternativas)
    precio_actual = None
    precio_antes = None

    # Número entero del precio actual
    actual_span = li.select_one(".poly-price__current .andes-money-amount__fraction")
    if actual_span:
        precio_actual = limpia_precio(actual_span.get_text())

    # Posible precio anterior (tachado) aparece con otra clase o data-test-id
    antes = (li.select_one(".andes-money-amount.comparative-price .andes-money-amount__fraction")
             or li.select_one("[data-test-id='price-before'] .andes-money-amount__fraction")
             or li.select_one(".ui-search-price__second-line .price-tag__disabled .price-tag-amount"))
    if antes:
        precio_antes = limpia_precio(antes.get_text())
    # -------- ATRIBUTOS NUEVOS (recámaras, baños, superficie) --------
    recamaras = None
    banos = None
    superficie_m2 = None

    attrs = li.select("ul.poly-attributes_list li.poly-attributes_list__item")
    for it in attrs:
        txt = it.get_text(" ", strip=True).lower()

        m = re.search(r"(\d+)\s*(recámaras|recamaras|habitaciones|dormitorios)", txt)
        if m:
            recamaras = int(m.group(1))
            continue

        m = re.search(r"(\d+)\s*(baños|banos)", txt)
        if m:
            banos = int(m.group(1))
            continue

        m = re.search(r"(\d+(?:\.\d+)?)\s*m²", txt)
        if m:
            superficie_m2 = float(m.group(1))
            continue

    return Articulo(
        nombre,
        descripcion,
        recamaras,
        banos,
        superficie_m2,
        precio_actual,
        link,
        imagen
    )

    return Articulo(nombre, descripcion,  precio_actual, link, imagen)

Articulo = namedtuple("Articulo", ["nombre", "descripcion", "recamaras", "banos", "superficie_m2", "precio_actual", "link", "imagen"])
articulos = []

# ---------- Config ----------
PAIS = "com.mx"  # cambia a "com.co", "com.ar", etc.

PAGINAS = 1            # cuántas páginas raspar
ESPERA_S = 30         # pausa entre páginas (ser buen ciudadano)
BASE = f"https://listado.mercadolibre.{PAIS}"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "es-MX,es;q=0.9"
}





In [5]:
# ---------- Scrape con paginado ----------
def scrap_page(PAGINAS,URL,HEADERS):
    for page in range(1, PAGINAS + 1):
        # En ML el paginado suele ser con _Desde_ o _Desde_XX; esta variante funciona en varios países:
        sep = "" if URL.endswith("/") else "/"
        url_pag = URL if page == 1 else f"{URL}{sep}_Desde_{(page-1)*48+1}"
        try:
            resp = requests.get(url_pag, headers=HEADERS, timeout=20)
            resp.raise_for_status()
        except requests.HTTPError as e:
            print(f"[WARN] {e} -> {url_pag}")
            break

        soup = BeautifulSoup(resp.text, "html.parser")

        # Cada item suele estar en <li> del layout de búsqueda
        items = soup.select("li.ui-search-layout__item, li.ui-search-layout__item--stack")
        if not items:
            # fallback genérico
            items = soup.select("li.ui-search-result__wrapper")
        if not items:
            print(f"[WARN] No se hallaron items en página {page}.")
            break

        for li in items:
            try:
                articulos.append(parse_item(li))
            except Exception as exc:
                # si un item rompe, sigue con el siguiente
                print(f"[WARN] item con error: {exc}")

        time.sleep(ESPERA_S)
        df = pd.DataFrame(articulos)
        latitudes = []
        longitudes = []
        
        for link in df["link"]:
            if link:
                lat, lon = get_lat_lon(link, HEADERS)
            else:
                lat, lon = None, None
        
            latitudes.append(lat)
            longitudes.append(lon)
        df["latitud"] = latitudes
        df["longitud"] = longitudes

        time.sleep(3)  # MUY importante para no bloquearte

        return df

In [6]:
#QUERY = list(dfp.product_name)[i]  # tu búsqueda
#URL = f"{BASE}/{QUERY.replace(' ', '-')}"
URL ='https://inmuebles.mercadolibre.com.mx/departamentos/venta/distrito-federal/coyoacan/'
articulos = []
print(URL)
df_scrap=scrap_page(3,URL,HEADERS)#[["nombre", "descripcion", "precio_actual"]]
print (f'se escrapearon {df_scrap.shape[0]} articulos')

https://inmuebles.mercadolibre.com.mx/departamentos/venta/distrito-federal/coyoacan/
se escrapearon 48 articulos


In [7]:
df_scrap

,nombre,descripcion,recamaras,banos,superficie_m2,precio_actual,link,imagen,latitud,longitud
0,None,"Departamentos Nuevos De Lujo En Ciudad Jardín,...",2.0,NaN,84.0,4300000,https://departamento.mercadolibre.com.mx/MLM-4...,https://http2.mlstatic.com/D_NQ_NP_2X_951017-M...,None,None
1,None,"Vivienda De Interes Social, Preventa, En Del C...",2.0,NaN,60.0,750000,https://departamento.mercadolibre.com.mx/MLM-2...,https://http2.mlstatic.com/D_NQ_NP_2X_611816-M...,None,None
2,None,"Depas Nuevos De Lujo En Campestre Churubusco, ...",2.0,2.0,74.0,4371708,https://departamento.mercadolibre.com.mx/MLM-4...,https://http2.mlstatic.com/D_NQ_NP_2X_906441-M...,None,None
3,None,Departamento En Venta En Insurgentes Cuicuilco,3.0,2.0,127.0,5000000,https://departamento.mercadolibre.com.mx/MLM-4...,https://http2.mlstatic.com/D_NQ_NP_2X_966192-M...,None,None
4,None,Preventa Pedregal De Santo Domingo,2.0,NaN,57.0,2607750,https://departamento.mercadolibre.com.mx/MLM-4...,https://http2.mlstatic.com/D_NQ_NP_2X_979717-M...,None,None
5,None,"Departamentos En Preventa En Coyoacán, Calzada...",NaN,NaN,33.0,2557000,https://departamento.mercadolibre.com.mx/MLM-2...,https://http2.mlstatic.com/D_NQ_NP_2X_831766-M...,None,None
6,None,"Departamento En Venta, Los Reyes Coyoacán, 3 R...",3.0,NaN,76.0,3000000,https://departamento.mercadolibre.com.mx/MLM-4...,https://http2.mlstatic.com/D_NQ_NP_2X_775674-M...,None,None
7,None,Departamento En Venta En El Reloj,3.0,3.0,138.0,6200000,https://departamento.mercadolibre.com.mx/MLM-2...,https://http2.mlstatic.com/D_NQ_NP_2X_626352-M...,None,None
8,None,"Venta Departamento En Ex Hacienda Coapa, Coyoa...",2.0,2.0,98.0,5950000,https://departamento.mercadolibre.com.mx/MLM-4...,https://http2.mlstatic.com/D_NQ_NP_2X_877953-M...,None,None
9,None,Venta Departamento En Coyoacán.,2.0,2.0,64.0,3890000,https://departamento.mercadolibre.com.mx/MLM-2...,https://http2.mlstatic.com/D_NQ_NP_2X_801581-M...,None,None


In [ ]:
url = "https://departamento.mercadolibre.com.mx/MLM-2687094089-departamento-en-venta-ctm-culhuacan-seccion-9-cdmx-_JM"
get_lat_lon(url, HEADERS)



In [ ]:
def get_desc(url, headers):
    r = requests.get(url, headers=headers, timeout=20)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    # 1) og:description (sale en tu captura)
    m = soup.select_one("meta[property='og:description']")
    if m and m.get("content"):
        desc = m["content"].strip()
        if desc:
            return desc

    # 2) twitter:description
    m = soup.select_one("meta[name='twitter:description']")
    if m and m.get("content"):
        desc = m["content"].strip()
        if desc:
            return desc

    # 3) descripción larga (DOM)
    p = soup.select_one("p.ui-pdp-description__content")
    if p:
        desc = p.get_text(" ", strip=True)
        if desc:
            return desc

    # 4) fallback por contenedor #description
    cont = soup.select_one("#description")
    if cont:
        # intenta agarrar el primer párrafo largo dentro
        p2 = cont.find("p")
        if p2:
            desc = p2.get_text(" ", strip=True)
            if desc:
                return desc

    return None


In [ ]:
url = "https://departamento.mercadolibre.com.mx/MLM-2687094089-departamento-en-venta-ctm-culhuacan-seccion-9-cdmx-_JM"
get_desc(url, HEADERS)


In [ ]:
# ---------- Scrape con paginado ----------
def scrap_page(PAGINAS,URL,HEADERS):
    for page in range(1, PAGINAS + 1):
        # En ML el paginado suele ser con _Desde_ o _Desde_XX; esta variante funciona en varios países:
        sep = "" if URL.endswith("/") else "/"
        url_pag = URL if page == 1 else f"{URL}{sep}_Desde_{(page-1)*48+1}"
        try:
            resp = requests.get(url_pag, headers=HEADERS, timeout=20)
            resp.raise_for_status()
        except requests.HTTPError as e:
            print(f"[WARN] {e} -> {url_pag}")
            break

        soup = BeautifulSoup(resp.text, "html.parser")

        # Cada item suele estar en <li> del layout de búsqueda
        items = soup.select("li.ui-search-layout__item, li.ui-search-layout__item--stack")
        if not items:
            # fallback genérico
            items = soup.select("li.ui-search-result__wrapper")
        if not items:
            print(f"[WARN] No se hallaron items en página {page}.")
            break

        for li in items:
            try:
                articulos.append(parse_item(li))
            except Exception as exc:
                # si un item rompe, sigue con el siguiente
                print(f"[WARN] item con error: {exc}")

        time.sleep(ESPERA_S)
        df = pd.DataFrame(articulos)
    latitudes = []
    longitudes = []
        
    col_link = "link"   # o "link" / "linf" según tu df_coyo
    filas = []
    
    for idx, url in enumerate(df[col_link]):
        print(f"Fila {idx} - URL: {url}")
    
        lat, lon = (None, None)
        desc = None
    
        if pd.isna(url) or not url:
            filas.append({"idx": idx, "latitud": None, "longitud": None, "descripcion": None})
            continue
    
        try:
            lat, lon = get_lat_lon(url, HEADERS)
        except Exception as e:
            print(f"[WARN] lat/lon falló en fila {idx}: {e}")
            lat, lon = (None, None)
    
        try:
            desc = get_desc(url, HEADERS)
        except Exception as e:
            print(f"[WARN] desc falló en fila {idx}: {e}")
            desc = None
    
        filas.append({
            "idx": idx,
            "latitud": lat,
            "longitud": lon,
            "descripcion": desc
        })
    
        # espera aleatoria 10–30s (como dijiste)
        time.sleep(random.uniform(10, 30))

        #time.sleep(3)  # MUY importante para no bloquearte
    df_extra = pd.DataFrame(filas).set_index("idx")

    df_final = pd.concat([df.reset_index(drop=True), df_extra], axis=1)
    df_final.head()
    print(df_final)
    return df_final

In [ ]:
#QUERY = list(dfp.product_name)[i]  # tu búsqueda
#URL = f"{BASE}/{QUERY.replace(' ', '-')}"
URL ='https://inmuebles.mercadolibre.com.mx/departamentos/venta/distrito-federal/coyoacan/'
articulos = []
print(URL)
df_scrap=scrap_page(1,URL,HEADERS)#[["nombre", "descripcion", "precio_actual"]]
print (f'se escrapearon {df_scrap.shape[0]} articulos')

In [ ]:
df_scrap

In [ ]:
import re
import json
import requests
from bs4 import BeautifulSoup

# -------- helpers --------
def _json_from_preloaded_state(html: str):
    soup = BeautifulSoup(html, "html.parser")
    s = soup.find("script", id="__PRELOADED_STATE__")
    if not s:
        return None
    raw = s.get_text(strip=False)
    m = re.search(r"\{.*\}", raw, flags=re.DOTALL)
    if not m:
        return None
    return json.loads(m.group(0))

def _find_first(obj, predicate):
    """Devuelve el primer dict/list element que cumpla predicate(obj)==True."""
    if predicate(obj):
        return obj
    if isinstance(obj, dict):
        for v in obj.values():
            found = _find_first(v, predicate)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for it in obj:
            found = _find_first(it, predicate)
            if found is not None:
                return found
    return None

def _find_all(obj, predicate, out=None):
    """Recolecta todos los elementos que cumplan predicate."""
    if out is None:
        out = []
    if predicate(obj):
        out.append(obj)
    if isinstance(obj, dict):
        for v in obj.values():
            _find_all(v, predicate, out)
    elif isinstance(obj, list):
        for it in obj:
            _find_all(it, predicate, out)
    return out

def _safe_float_m2(x):
    if x is None:
        return None
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).lower().replace(",", ".")
    m = re.search(r"(\d+(?:\.\d+)?)", s)
    return float(m.group(1)) if m else None

def _attr_value(a):
    """Saca el valor usable de un attribute dict típico de ML."""
    if not isinstance(a, dict):
        return None
    # prioridades comunes
    if "value_number" in a and a["value_number"] is not None:
        return a["value_number"]
    if "value_name" in a and a["value_name"] is not None:
        return a["value_name"]
    if "value_struct" in a and isinstance(a["value_struct"], dict):
        # a veces viene {number: X, unit: 'm²'}
        vs = a["value_struct"]
        if "number" in vs and vs["number"] is not None:
            return vs["number"]
        if "value" in vs and vs["value"] is not None:
            return vs["value"]
    return None

# -------- función principal --------
def get_listing_detail(url: str, headers: dict):
    """
    Extrae info del inmueble desde __PRELOADED_STATE__:
    coords, precio, titulo, descripción (si viene), atributos estructurados.
    """
    # normaliza fragmento
    url = url.split("#")[0].strip()

    r = requests.get(url, headers=headers, timeout=20)
    r.raise_for_status()

    data = _json_from_preloaded_state(r.text)
    if data is None:
        return None

    out = {
        "url": url,
        "titulo": None,
        "precio": None,
        "moneda": None,
        "latitud": None,
        "longitud": None,
        "direccion_texto": None,
        "recamaras": None,
        "banos": None,
        "sup_total_m2": None,
        "sup_construida_m2": None,
        "estacionamientos": None,
        "descripcion": None,
        "attributes_raw": None,
    }

    # 1) Coordenadas: buscamos cualquier dict que tenga map_info.location.latitude/longitude
    def pred_mapinfo(obj):
        return (
            isinstance(obj, dict)
            and "map_info" in obj
            and isinstance(obj.get("map_info"), dict)
            and isinstance(obj["map_info"].get("location"), dict)
            and ("latitude" in obj["map_info"]["location"])
            and ("longitude" in obj["map_info"]["location"])
        )

    mi = _find_first(data, pred_mapinfo)
    if isinstance(mi, dict):
        loc = mi["map_info"]["location"]
        out["latitud"] = loc.get("latitude")
        out["longitud"] = loc.get("longitude")

    # 2) Buscar "item" / "listing" candidato: dict que tenga attributes list
    def pred_item(obj):
        return isinstance(obj, dict) and isinstance(obj.get("attributes"), list) and len(obj["attributes"]) > 0

    item = _find_first(data, pred_item)

    # 3) Título / precio / moneda / dirección (si están dentro del item)
    if isinstance(item, dict):
        out["titulo"] = item.get("title") or item.get("headline") or item.get("name")

        # precio: distintas variantes
        price = item.get("price")
        if isinstance(price, dict):
            out["precio"] = price.get("amount") or price.get("value") or price.get("number")
            out["moneda"] = price.get("currency_id") or price.get("currency")
        else:
            # a veces está como number suelto
            out["precio"] = item.get("price") if isinstance(item.get("price"), (int, float)) else out["precio"]
            out["moneda"] = item.get("currency_id") or out["moneda"]

        # dirección textual (cuando existe)
        # buscamos campos típicos
        out["direccion_texto"] = (
            item.get("location", {}).get("address_line")
            if isinstance(item.get("location"), dict)
            else None
        )
        if out["direccion_texto"] is None and isinstance(item.get("location"), dict):
            # a veces viene "name" o "title" del location
            out["direccion_texto"] = item["location"].get("title") or item["location"].get("name")

        out["attributes_raw"] = item.get("attributes")

        # 4) Mapear attributes a tus columnas
        attrs = item.get("attributes", [])
        id_to_val = {}
        for a in attrs:
            if isinstance(a, dict) and a.get("id"):
                id_to_val[a["id"]] = _attr_value(a)

        # ids comunes en inmuebles ML (pueden variar, por eso probamos varios)
        # recámaras
        for k in ["BEDROOMS", "ROOMS", "DORMITORIES"]:
            if k in id_to_val and id_to_val[k] is not None:
                out["recamaras"] = int(float(str(id_to_val[k]).strip())) if str(id_to_val[k]).strip().replace(".","",1).isdigit() else id_to_val[k]
                break

        # baños
        for k in ["FULL_BATHROOMS", "BATHROOMS"]:
            if k in id_to_val and id_to_val[k] is not None:
                out["banos"] = int(float(str(id_to_val[k]).strip())) if str(id_to_val[k]).strip().replace(".","",1).isdigit() else id_to_val[k]
                break

        # superficies
        for k in ["TOTAL_AREA", "LOT_AREA"]:
            if k in id_to_val and id_to_val[k] is not None:
                out["sup_total_m2"] = _safe_float_m2(id_to_val[k])
                break

        for k in ["COVERED_AREA", "PROPERTY_AREA"]:
            if k in id_to_val and id_to_val[k] is not None:
                out["sup_construida_m2"] = _safe_float_m2(id_to_val[k])
                break

        # estacionamientos
        for k in ["PARKING_LOTS", "GARAGES"]:
            if k in id_to_val and id_to_val[k] is not None:
                out["estacionamientos"] = int(float(str(id_to_val[k]).strip())) if str(id_to_val[k]).strip().replace(".","",1).isdigit() else id_to_val[k]
                break

    # 5) Descripción: a veces viene como plain_text en otra rama del JSON
    # buscamos cualquier dict con clave 'plain_text' (típico en ML)
    def pred_plain(obj):
        return isinstance(obj, dict) and isinstance(obj.get("plain_text"), str) and len(obj["plain_text"]) > 30

    desc_obj = _find_first(data, pred_plain)
    if isinstance(desc_obj, dict):
        out["descripcion"] = desc_obj.get("plain_text")

    # fallback por si viene como "description" string largo
    if out["descripcion"] is None:
        def pred_desc(obj):
            return isinstance(obj, dict) and isinstance(obj.get("description"), str) and len(obj["description"]) > 60
        d2 = _find_first(data, pred_desc)
        if isinstance(d2, dict):
            out["descripcion"] = d2.get("description")

    return out


In [ ]:
alcaldias = [
"alvaro-obregon",
"azcapotzalco",
"benito-juarez",
"coyoacan",
"cuajimalpa-de-morelos",
"cuauhtemoc",
"gustavo-a-madero",
"iztacalco",
"iztapalapa",
"la-magdalena-contreras",
"miguel-hidalgo",
"milpa-alta",
"tlahuac",
"tlalpan",
"venustiano-carranza",
"xochimilco"
]

tipos = ["casas", "departamentos"]
operaciones = ["venta", "renta"]

base = "https://inmuebles.mercadolibre.com.mx"

urls = [
f"{base}/{tipo}/{operacion}/distrito-federal/{alcaldia}/"
for alcaldia in alcaldias
for tipo in tipos
for operacion in operaciones
]

print(urls)

In [ ]:
edomex = [
"atizapan-de-zaragoza",
"coacalco-de-berriozabal",
"cuautitlan",
"cuautitlan-izcalli",
"chalco",
"chicoloapan",
"chimalhuacan",
"ecatepec-de-morelos",
"huixquilucan",
"ixtapaluca",
"la-paz",
"naucalpan-de-juarez",
"nezahualcoyotl",
"nicolas-romero",
"tecamac",
"teoloyucan",
"tepotzotlan",
"texcoco",
"tlalnepantla-de-baz",
"tultitlan",
"valle-de-chalco-solidaridad",
"zumpango"
]

urls += [
f"{base}/{tipo}/{operacion}/estado-de-mexico/{mun}/"
for mun in edomex
for tipo in tipos
for operacion in operaciones
]

In [ ]:
urls

In [ ]:
base = "https://inmuebles.mercadolibre.com.mx"

tipos = ["casas", "departamentos"]
operaciones = ["venta", "renta"]

# -------------------------
# 1) CDMX: 16 alcaldías
# -------------------------
cdmx_alcaldias = [
    "alvaro-obregon",
    "azcapotzalco",
    "benito-juarez",
    "coyoacan",
    "cuajimalpa-de-morelos",
    "cuauhtemoc",
    "gustavo-a-madero",
    "iztacalco",
    "iztapalapa",
    "la-magdalena-contreras",
    "miguel-hidalgo",
    "milpa-alta",
    "tlahuac",
    "tlalpan",
    "venustiano-carranza",
    "xochimilco",
]

# -------------------------
# 2) EdoMex metropolitano
# -------------------------
edomex_metro = [
    "atizapan-de-zaragoza",
    "coacalco-de-berriozabal",
    "cuautitlan",
    "cuautitlan-izcalli",
    "chalco",
    "chicoloapan",
    "chimalhuacan",
    "ecatepec-de-morelos",
    "huixquilucan",
    "ixtapaluca",
    "la-paz",
    "naucalpan-de-juarez",
    "nezahualcoyotl",
    "nicolas-romero",
    "tecamac",
    "teoloyucan",
    "tepotzotlan",
    "texcoco",
    "tlalnepantla-de-baz",
    "tultitlan",
    "valle-de-chalco-solidaridad",
    "zumpango",
]

# -------------------------
# 3) Estados a nivel general
# -------------------------
estados_generales = [
    "aguascalientes",
    "baja-california",
    "baja-california-sur",
    "campeche",
    "chiapas",
    "chihuahua",
    "coahuila",
    "colima",
    "durango",
    "guanajuato",
    "guerrero",
    "hidalgo",
    "jalisco",
    "michoacan",
    "morelos",
    "nayarit",
    "nuevo-leon",
    "oaxaca",
    "puebla",
    "queretaro",
    "quintana-roo",
    "san-luis-potosi",
    "sinaloa",
    "sonora",
    "tabasco",
    "tamaulipas",
    "tlaxcala",
    "veracruz",
    "yucatan",
    "zacatecas",
]

# -------------------------
# 4) Ciudades / municipios grandes
#    (para scraping más fino)
# -------------------------
ciudades_grandes = {
    "aguascalientes": [
        "aguascalientes",
        "jesus-maria",
    ],
    "baja-california": [
        "tijuana",
        "mexicali",
        "ensenada",
        "tecate",
        "playas-de-rosarito",
    ],
    "baja-california-sur": [
        "la-paz",
        "los-cabos",
    ],
    "campeche": [
        "campeche",
        "carmen",
    ],
    "chiapas": [
        "tuxtla-gutierrez",
        "san-cristobal-de-las-casas",
        "tapachula",
    ],
    "chihuahua": [
        "chihuahua",
        "juarez",
        "delicias",
    ],
    "coahuila": [
        "saltillo",
        "torreon",
        "monclova",
        "piedras-negras",
        "ramos-arizpe",
    ],
    "colima": [
        "colima",
        "manzanillo",
        "villa-de-alvarez",
    ],
    "durango": [
        "durango",
        "gomez-palacio",
        "lerdo",
    ],
    "guanajuato": [
        "leon",
        "celaya",
        "irapuato",
        "salamanca",
        "guanajuato",
        "san-miguel-de-allende",
    ],
    "guerrero": [
        "acapulco-de-juarez",
        "chilpancingo-de-los-bravo",
        "iguala-de-la-independencia",
    ],
    "hidalgo": [
        "pachuca-de-soto",
        "mineral-de-la-reforma",
        "tizayuca",
        "tula-de-allende",
    ],
    "jalisco": [
        "guadalajara",
        "zapopan",
        "tlajomulco-de-zuniga",
        "san-pedro-tlaquepaque",
        "tonala",
        "puerto-vallarta",
    ],
    "michoacan": [
        "morelia",
        "uruapan",
        "zamora",
        "lazaro-cardenas",
    ],
    "morelos": [
        "cuernavaca",
        "jiutepec",
        "temixco",
        "yautepec",
        "emiliano-zapata",
        "xochitepec",
        "cuautla",
    ],
    "nayarit": [
        "tepic",
        "bahia-de-banderas",
    ],
    "nuevo-leon": [
        "monterrey",
        "guadalupe",
        "san-nicolas-de-los-garza",
        "apodaca",
        "general-escobedo",
        "santa-catarina",
        "san-pedro-garza-garcia",
        "garcia",
        "juarez",
    ],
    "oaxaca": [
        "oaxaca-de-juarez",
        "santa-cruz-xoxocotlan",
        "san-juan-bautista-tuxtepec",
    ],
    "puebla": [
        "puebla",
        "san-andres-cholula",
        "san-pedro-cholula",
        "cuautlancingo",
        "tehuacan",
        "coronango",
        "atlixco",
    ],
    "queretaro": [
        "queretaro",
        "el-marques",
        "corregidora",
        "san-juan-del-rio",
    ],
    "quintana-roo": [
        "benito-juarez",
        "solidaridad",
        "tulum",
        "othon-p-blanco",
        "isla-mujeres",
        "puerto-morelos",
    ],
    "san-luis-potosi": [
        "san-luis-potosi",
        "soledad-de-graciano-sanchez",
    ],
    "sinaloa": [
        "culiacan",
        "mazatlan",
        "ahome",
        "guasave",
    ],
    "sonora": [
        "hermosillo",
        "cajeme",
        "nogales",
        "san-luis-rio-colorado",
    ],
    "tabasco": [
        "centro",
        "paraiso",
        "comalcalco",
    ],
    "tamaulipas": [
        "reynosa",
        "matamoros",
        "nuevo-laredo",
        "tampico",
        "ciudad-madero",
        "altamira",
        "victoria",
    ],
    "tlaxcala": [
        "tlaxcala",
        "apizaco",
        "chiautempan",
        "huamantla",
    ],
    "veracruz": [
        "veracruz",
        "boca-del-rio",
        "xalapa",
        "coatzacoalcos",
        "cordoba",
        "orizaba",
        "poza-rica-de-hidalgo",
    ],
    "yucatan": [
        "merida",
        "progreso",
        "valladolid",
    ],
    "zacatecas": [
        "zacatecas",
        "guadalupe",
        "fresnillo",
    ],
}

# -------------------------
# Generación de URLs
# -------------------------
urls = []

# CDMX
urls += [
    f"{base}/{tipo}/{operacion}/distrito-federal/{alcaldia}/"
    for alcaldia in cdmx_alcaldias
    for tipo in tipos
    for operacion in operaciones
]

# EdoMex metropolitano
urls += [
    f"{base}/{tipo}/{operacion}/estado-de-mexico/{municipio}/"
    for municipio in edomex_metro
    for tipo in tipos
    for operacion in operaciones
]

# Estados generales
urls += [
    f"{base}/{tipo}/{operacion}/{estado}/"
    for estado in estados_generales
    for tipo in tipos
    for operacion in operaciones
]

# Ciudades / municipios grandes
urls += [
    f"{base}/{tipo}/{operacion}/{estado}/{ciudad}/"
    for estado, ciudades in ciudades_grandes.items()
    for ciudad in ciudades
    for tipo in tipos
    for operacion in operaciones
]

# quitar duplicados por si algo se repite
urls = list(dict.fromkeys(urls))

print(f"Total URLs: {len(urls)}")
print(urls[:20])